# INF2006 G12 — Task 3: Next-week volatility category (FINAL candidate notebook)
This notebook is **self-contained**: load the existing Task 2 S3 tables, create past-only features, compare a few models, choose the probability cutoff using 2016 validation data, evaluate on 2017 test data once, save the final pipeline and verify reloading.

**How to run (EMR JupyterHub, PySpark kernel):**
1. Run all cells from the top (Kernel → Restart & Run All). The last cell will print *SKIPPED* because it is still the training session; that is expected.
2. Note the `RUN_ID` printed by the save cell. Then **Kernel → Restart** (a new Spark session starts) and run **only the last cell**. It reloads the saved model from S3 without retraining and compares all saved 2017 test predictions against a tolerance of 1e-9.
3. File → Save, then File → Download as .ipynb so the outputs are stored in the notebook.

NumPy is installed on every node by the `install-numpy.sh` bootstrap action.

**Meaning of class 1:** next-five-session volatility exceeds the threshold used to create the existing volatility label. In both notebooks, the model predicts a **class**, not an exact future price or numerical volatility. The probability score estimates the likelihood of class 1.

**Important:** Only use information from the current or earlier date as a feature. The test split is for the final one-time assessment, not repeated tuning.


In [ ]:
# CELL 1 — Settings
# We use time splits: 2013–2015 train, 2016 validation, 2017 test.
import time, json, datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

spark.conf.set("spark.sql.shuffle.partitions", "32")  # fewer tiny Spark tasks on small EMR clusters
BUCKET = "inf2006-g12-ml"
BASE = "s3a://" + BUCKET
PRICES_PATH = BASE + "/task2/cleaned/cleaned/prices"
TASK = "volatility"
INPUT = BASE + ("/task2/model_ready_v1" if TASK == "direction" else "/task3/volatility_v1")
LABEL = "label" if TASK == "direction" else "label_volatility_5d"
SPLITS = ["training", "validation", "testing"]
BASE_FEATURES = ["daily_return", "return_5d", "volatility_20d", "volume_change_1d", "intraday_range"]
EXTRA_FEATURES = ["volatility_5d", "volatility_60d", "vol_ratio_5_60", "abs_return_5d", "range_5d", "return_20d", "volume_z_20d", "market_return_1d", "market_volatility_5d"]
ALL_FEATURES = BASE_FEATURES + EXTRA_FEATURES
SEED = 42
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = BASE + "/task3final/volatility/" + RUN_ID
# Three sensible candidates. Start with these before trying expensive huge searches.
# Switch the fourth model on after the first successful complete run, if needed.
TRY_RANDOM_FOREST = False
print("Task:", TASK, "| Spark:", spark.version, "| RUN_ID:", RUN_ID)
print("Input:", INPUT, "| Save location:", OUT)


In [ ]:
# Load and check Task 2's existing labels and five original input features.
# An input feature describes what we know as of TODAY. The label describes the FUTURE.
def prepare(df):
    required = ["ticker", "trade_date", LABEL] + BASE_FEATURES
    missing = sorted(set(required) - set(df.columns))
    assert not missing, "Missing columns: " + str(missing)
    out = df.select("ticker", F.to_date("trade_date").alias("trade_date"),
        *[F.col(c).cast("double").alias(c) for c in BASE_FEATURES],
        F.col(LABEL).cast("double").alias("target"))
    invalid = F.col("target").isNull() | (~F.col("target").isin(0.0, 1.0))
    for col in BASE_FEATURES:
        invalid = invalid | F.col(col).isNull() | F.isnan(col) | (F.abs(F.col(col)) == float("inf"))
    assert out.filter(invalid).limit(1).count() == 0, "Invalid original features or target"
    assert out.groupBy("ticker", "trade_date").count().filter("count > 1").limit(1).count() == 0, "Duplicate ticker/date"
    return out

data = {}
for split in SPLITS:
    data[split] = prepare(spark.read.parquet(INPUT + "/" + split)).cache()
    n = data[split].count()
    summary = data[split].agg(F.min("trade_date"), F.max("trade_date"), F.avg("target")).first()
    print(split, "rows:", n, "dates:", summary[0], "to", summary[1], "positive share:", round(summary[2], 4))
# The row counts below were observed in your earlier completed notebooks.
expected = ({"training":349590, "validation":122801, "testing":126001} if TASK == "direction"
            else {"training":349590, "validation":122801, "testing":123476})
for split in SPLITS:
    assert data[split].count() == expected[split], "Unexpected " + split + " count; investigate changed upstream data"
print("Input data checks passed.")


In [ ]:
# Build extra features using ONLY current and earlier daily prices.
prices = spark.read.parquet(PRICES_PATH).select("ticker", F.to_date("trade_date").alias("trade_date"),
    *[F.col(c).cast("double").alias(c) for c in ["high_price", "low_price", "close_price", "volume"]])
assert prices.groupBy("ticker", "trade_date").count().filter("count > 1").limit(1).count() == 0, "Duplicate cleaned prices"
w = Window.partitionBy("ticker").orderBy("trade_date")
def past(days):
    return w.rowsBetween(-(days - 1), 0)  # never reads future dates
p = (prices.withColumn("prior_close", F.lag("close_price").over(w))
    .withColumn("ret", F.when(F.col("prior_close") > 0, F.col("close_price") / F.col("prior_close") - 1))
    .withColumn("rng", F.when(F.col("close_price") > 0, (F.col("high_price") - F.col("low_price")) / F.col("close_price")))
    .withColumn("logvol", F.when(F.col("volume") >= 0, F.log1p("volume"))))
p = (p.withColumn("volatility_5d", F.stddev_samp("ret").over(past(5)))
    .withColumn("volatility_60d", F.stddev_samp("ret").over(past(60)))
    .withColumn("abs_return_5d", F.avg(F.abs("ret")).over(past(5)))
    .withColumn("range_5d", F.avg("rng").over(past(5)))
    .withColumn("return_20d", F.when(F.lag("close_price", 20).over(w) > 0,
        F.col("close_price") / F.lag("close_price", 20).over(w) - 1))
    .withColumn("volume_z_20d", F.when(F.stddev_samp("logvol").over(past(20)) > 0,
        (F.col("logvol") - F.avg("logvol").over(past(20))) / F.stddev_samp("logvol").over(past(20)))))
p = p.withColumn("vol_ratio_5_60", F.when(F.col("volatility_60d") > 0,
    F.col("volatility_5d") / F.col("volatility_60d")))
market = Window.partitionBy("trade_date")
p = (p.withColumn("market_return_1d", F.avg("ret").over(market))
      .withColumn("market_volatility_5d", F.avg("volatility_5d").over(market)))
extra = p.select("ticker", "trade_date", *EXTRA_FEATURES)
for c in EXTRA_FEATURES:
    extra = extra.withColumn(c, F.when(F.isnan(c) | (F.abs(F.col(c)) == float("inf")), F.lit(None).cast("double")).otherwise(F.col(c)))
# Missing *extra* values are handled by an Imputer fitted ONLY on training.
# Left join deliberately keeps rows that have no matching cleaned-price record.
for split in SPLITS:
    old = data[split]
    joined = old.join(extra, ["ticker", "trade_date"], "left").cache()
    assert joined.count() == old.count(), "Feature join unexpectedly changed the number of rows"
    data[split] = joined
    old.unpersist()
print("Extra features added:", EXTRA_FEATURES)
print("Missing extra features in training (model will fill them from training medians):")
data["training"].select(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in EXTRA_FEATURES]).show(vertical=True)


In [ ]:
# How we judge models
# Balanced accuracy gives equal importance to detecting 0 and detecting 1.
# AUC measures how well the probability scores rank positive examples above negative ones.
auc_evaluator = BinaryClassificationEvaluator(labelCol="target", rawPredictionCol="risk_score", metricName="areaUnderROC")
def measure(df, title, use_auc=True):
    counts = {(int(r["target"]), int(r["prediction"])): r["count"]
              for r in df.groupBy("target", "prediction").count().collect()}
    tn, fp, fn, tp = [counts.get(key, 0) for key in [(0,0), (0,1), (1,0), (1,1)]]
    total = tn + fp + fn + tp
    assert total > 0, "No data to evaluate"
    recall = tp / (tp + fn) if tp + fn else 0.0
    specificity = tn / (tn + fp) if tn + fp else 0.0
    precision = tp / (tp + fp) if tp + fp else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    result = dict(model=title, rows=total, auc=round(float(auc_evaluator.evaluate(df)), 5) if use_auc else None,
        accuracy=round((tp + tn) / total, 5), balanced_accuracy=round((recall + specificity) / 2, 5),
        f1=round(f1, 5), recall=round(recall, 5), precision=round(precision, 5),
        tn=tn, fp=fp, fn=fn, tp=tp)
    print(title, result)
    return result

# Faster than evaluating hundreds of full Spark queries: collect only a SMALL score histogram.
# Search cutoff 0.05–0.95 in steps of 0.01. We choose this on VALIDATION, never on TEST.
def find_threshold(df):
    hist = (df.select(F.least(F.greatest(F.floor(F.col("risk_score") * 100), F.lit(0)), F.lit(100)).cast("int").alias("bin"), "target")
              .groupBy("bin", "target").count().collect())
    positives, negatives = [0] * 101, [0] * 101
    for r in hist:
        (positives if float(r["target"]) == 1.0 else negatives)[int(r["bin"])] += r["count"]
    P, N = sum(positives), sum(negatives)
    assert P > 0 and N > 0, "Both classes must appear in validation"
    best = (-1.0, None)
    for k in range(5, 96):
        tp = sum(positives[k:])   # predict 1 when probability >= k / 100
        tn = sum(negatives[:k])   # predict 0 otherwise
        bal = (tp / P + tn / N) / 2
        if bal > best[0]:
            best = (bal, k / 100.0)
    return best[1]

def with_cutoff(df, threshold):
    return df.withColumn("prediction", (F.col("risk_score") >= F.lit(float(threshold))).cast("double"))

# Simple rule baselines let us check whether ML actually beats an easy alternative.
majority = float(data["training"].groupBy("target").count().orderBy(F.desc("count")).first()["target"])
rule_col = "return_5d" if TASK == "direction" else "volatility_20d"
rule_cutoff = 0.0 if TASK == "direction" else float(data["training"].approxQuantile(rule_col, [0.75], 0.001)[0])
def evaluate_baselines(df, split):
    m = measure(df.withColumn("prediction", F.lit(majority)).withColumn("risk_score", F.lit(majority)),
                "always_majority_" + split)
    r = measure(df.withColumn("prediction", (F.col(rule_col) > rule_cutoff).cast("double"))
                .withColumn("risk_score", F.col(rule_col)), "historical_rule_" + split)
    return [m, r]
validation_baselines = evaluate_baselines(data["validation"], "validation")
print("Rule:", rule_col, ">", rule_cutoff)


In [ ]:
# Train several CANDIDATES on training data; compare using 2016 validation.
# Logistic regression gets scaled data; tree models use unscaled data (faster).
def pipeline_for(kind, cols):
    imputed = [c + "_filled" for c in cols]
    steps = [Imputer(inputCols=cols, outputCols=imputed, strategy="median")]
    if kind == "lr":
        steps += [VectorAssembler(inputCols=imputed, outputCol="unscaled"),
                  StandardScaler(inputCol="unscaled", outputCol="features", withStd=True, withMean=False),
                  LogisticRegression(featuresCol="features", labelCol="target", maxIter=100, regParam=0.01)]
    elif kind == "gbt":
        steps += [VectorAssembler(inputCols=imputed, outputCol="features"),
                  GBTClassifier(featuresCol="features", labelCol="target", maxIter=35, maxDepth=4,
                                minInstancesPerNode=30, seed=SEED)]
    elif kind == "rf":
        steps += [VectorAssembler(inputCols=imputed, outputCol="features"),
                  RandomForestClassifier(featuresCol="features", labelCol="target", numTrees=50,
                                         maxDepth=7, minInstancesPerNode=30, seed=SEED)]
    return Pipeline(stages=steps)

candidates = [("logistic_base", "lr", BASE_FEATURES),
              ("logistic_extra", "lr", ALL_FEATURES),
              ("gbt_extra", "gbt", ALL_FEATURES)]
if TRY_RANDOM_FOREST:
    candidates.append(("random_forest_extra", "rf", ALL_FEATURES))
results = []
fitted = {}
for name, kind, cols in candidates:
    start = time.time()
    print("\nTRAINING:", name, "with", len(cols), "features", flush=True)
    fitted[name] = pipeline_for(kind, cols).fit(data["training"])
    scores = (fitted[name].transform(data["validation"])
        .withColumn("risk_score", vector_to_array("probability")[1]).select("target", "risk_score").cache())
    scores.count()  # materialize cache once, then reuse for threshold + evaluation
    threshold = find_threshold(scores)
    comparison = measure(with_cutoff(scores, threshold), name + "_validation")
    comparison.update(name=name, threshold=threshold, feature_count=len(cols),
                      fit_seconds=round(time.time() - start, 1))
    results.append(comparison)
    scores.unpersist()
    print("Threshold selected on VALIDATION:", threshold)
# Prioritize ranking quality (AUC); balanced accuracy breaks ties.
# The threshold is separately tuned to make decisions about class 0 versus class 1.
chosen = sorted(results, key=lambda x: (-x["auc"], -x["balanced_accuracy"], x["name"]))[0]
chosen_name = chosen["name"]
chosen_threshold = chosen["threshold"]
final_model = fitted[chosen_name]
chosen_features = next(cols for name, kind, cols in candidates if name == chosen_name)
print("\nWINNING MODEL:", chosen_name, "| validation AUC:", chosen["auc"],
      "| validation balanced accuracy:", chosen["balanced_accuracy"], "| decision cutoff:", chosen_threshold)
print("Comparison:")
for r in results:
    print(r["name"], "AUC", r["auc"], "balanced accuracy", r["balanced_accuracy"],
          "F1", r["f1"], "threshold", r["threshold"], "fit seconds", r["fit_seconds"])
print("Test data has NOT been used to choose the model.")


In [ ]:
# Bigger tree importance = this feature influenced splits more. Not proof of causation.
# Logistic coefficients refer to standardized features (sign tells direction of association).
model_stage = final_model.stages[-1]
if hasattr(model_stage, "coefficients"):
    weights = model_stage.coefficients.toArray()
    meaning = "standardized logistic coefficient"
else:
    weights = model_stage.featureImportances.toArray()
    meaning = "tree feature importance"
print("Chosen algorithm:", chosen_name, "|", meaning)
for col, weight in sorted(zip(chosen_features, weights), key=lambda pair: -abs(pair[1])):
    print("{:<26s} {: .5f}".format(col, float(weight)))
print("WARNING: feature importance is not proof of prediction reliability or causality.")


In [ ]:
# Final one-time evaluation on unseen 2017 data.

test_scores = (final_model.transform(data["testing"])
    .withColumn("risk_score", vector_to_array("probability")[1]))
test_predictions = with_cutoff(test_scores, chosen_threshold).cache()
test_predictions.count()
final_test = measure(test_predictions, chosen_name + "_FINAL_TEST")
test_baselines = evaluate_baselines(data["testing"], "testing")
print("\nCONFUSION MATRIX (actual rows, predicted columns):")
print("             predict 0   predict 1")
print("actual 0    {:>10} {:>10}".format(final_test["tn"], final_test["fp"]))
print("actual 1    {:>10} {:>10}".format(final_test["fn"], final_test["tp"]))
print("Compare balanced accuracy and AUC against the baselines, not accuracy alone.")


In [ ]:
# Save the pipeline, predictions and scores in S3.
# Note: Spark writes folders containing part-files, not one .csv file.
MODEL_PATH = OUT + "/model"
PRED_PATH = OUT + "/predictions"
METRICS_PATH = OUT + "/metrics"
SETTINGS_PATH = OUT + "/settings"
TEST_FEATURES_PATH = OUT + "/test_features"
final_model.write().overwrite().save(MODEL_PATH)
(test_predictions.select("ticker", "trade_date", "target", "prediction", "risk_score")
    .write.mode("overwrite").parquet(PRED_PATH))
# Raw (un-imputed) 2017 inputs, so a fresh session can re-score them with the saved model only.
data["testing"].select("ticker", "trade_date", "target", *chosen_features) \
    .write.mode("overwrite").parquet(TEST_FEATURES_PATH)
all_metrics = ([dict(stage="validation", **r) for r in validation_baselines + results] +
               [dict(stage="testing", **r) for r in test_baselines + [final_test]])
spark.createDataFrame([(json.dumps(r),) for r in all_metrics], ["value"]).coalesce(1).write.mode("overwrite").text(METRICS_PATH)
info = dict(task=TASK, run_id=RUN_ID, algorithm=chosen_name, threshold=chosen_threshold,
    features=chosen_features, input_path=INPUT, prices_path=PRICES_PATH,
    model_path=MODEL_PATH, predictions_path=PRED_PATH, test_features_path=TEST_FEATURES_PATH,
    spark_application_id=spark.sparkContext.applicationId, validation_auc=chosen["auc"],
    validation_balanced_accuracy=chosen["balanced_accuracy"])
spark.createDataFrame([(json.dumps(info),)], ["value"]).coalesce(1).write.mode("overwrite").text(SETTINGS_PATH)
assert spark.read.parquet(PRED_PATH).count() == data["testing"].count()
print("SAVED. RUN_ID:", RUN_ID)
print("MODEL:", MODEL_PATH)
print("PREDICTIONS:", PRED_PATH)
print("SETTINGS:", SETTINGS_PATH)
print("TEST FEATURES:", TEST_FEATURES_PATH)
print("RUN_ID:", RUN_ID, "| Spark application:", spark.sparkContext.applicationId)


## Fresh-session reload check
**Kernel → Restart first**, then run only the cell below. It must show a different Spark application ID from the training run.

In [ ]:
# FRESH-SESSION RELOAD CHECK (project brief 3.3)
# Run this cell ONLY after Kernel -> Restart. Nothing is retrained: it loads the saved
# PipelineModel from S3, re-scores the saved 2017 test inputs, and compares EVERY row
# with the predictions saved by the training session.
import json, datetime
from pyspark.sql import functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

TASK_DIR = "s3a://inf2006-g12-ml/task3final/volatility"  # one sub-folder per RUN_ID
RUN_ID = ""          # blank = newest saved run; or paste the RUN_ID printed by the save cell
TOLERANCE = 1e-9     # maximum allowed absolute difference in class-1 probability

Path = spark._jvm.org.apache.hadoop.fs.Path
root = Path(TASK_DIR)
fs = root.getFileSystem(spark._jsc.hadoopConfiguration())
if not RUN_ID:
    runs = sorted(st.getPath().getName() for st in fs.listStatus(root)
                  if st.isDirectory() and fs.exists(Path(TASK_DIR + "/" + st.getPath().getName() + "/test_features")))
    assert runs, "No saved run with test_features under " + TASK_DIR + " - run the notebook from the top first."
    RUN_ID = runs[-1]
RUN = TASK_DIR + "/" + RUN_ID
cfg = json.loads(spark.read.text(RUN + "/settings").first()["value"])
cutoff = float(cfg.get("cutoff", cfg.get("threshold")))
this_app = spark.sparkContext.applicationId
print("RUN_ID:", RUN_ID, "| trained in Spark app", cfg.get("spark_application_id"), "| this session:", this_app)

if cfg.get("spark_application_id") == this_app:
    print("SKIPPED: this is still the training session. Do Kernel -> Restart, then run ONLY this cell.")
else:
    for part in ["model", "test_features", "predictions", "settings"]:
        assert fs.exists(Path(RUN + "/" + part)), "MISSING saved files: " + RUN + "/" + part
    model = PipelineModel.load(RUN + "/model")          # loaded, not fitted
    print("Loaded stages:", [type(s).__name__ for s in model.stages])
    features = spark.read.parquet(RUN + "/test_features")
    saved = spark.read.parquet(RUN + "/predictions")
    new = (model.transform(features)
           .withColumn("risk_new", vector_to_array("probability")[1])
           .withColumn("pred_new", (F.col("risk_new") >= F.lit(cutoff)).cast("double"))
           .select("ticker", "trade_date", "risk_new", "pred_new"))
    both = saved.join(new, ["ticker", "trade_date"], "full_outer")
    s = both.agg(F.count(F.lit(1)).alias("rows_compared"),
        F.sum(F.col("risk_score").isNull().cast("int")).alias("rows_only_in_reload"),
        F.sum(F.col("risk_new").isNull().cast("int")).alias("rows_only_in_saved"),
        F.max(F.abs(F.col("risk_score") - F.col("risk_new"))).alias("max_abs_prob_diff"),
        F.sum((F.col("prediction") != F.col("pred_new")).cast("int")).alias("class_mismatches")).first().asDict()
    s["saved_prediction_rows"] = saved.count()
    passed = (s["rows_only_in_reload"] == 0 and s["rows_only_in_saved"] == 0 and s["class_mismatches"] == 0
              and s["max_abs_prob_diff"] is not None and s["max_abs_prob_diff"] <= TOLERANCE)
    record = dict(check="fresh-session saved-model reload", task=cfg.get("task"), run_id=RUN_ID,
                  model_path=RUN + "/model", cutoff=cutoff, tolerance=TOLERANCE,
                  checked_at_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
                  training_spark_app=cfg.get("spark_application_id"), reload_spark_app=this_app,
                  passed=passed, **s)
    spark.createDataFrame([(json.dumps(record, default=str),)], ["value"]).coalesce(1) \
         .write.mode("overwrite").text(RUN + "/reload_check")
    print(json.dumps(record, indent=1, default=str))
    print("\nRELOAD CHECK", "PASSED" if passed else "FAILED", "-", s["saved_prediction_rows"],
          "test rows compared, tolerance", TOLERANCE, "| record saved to", RUN + "/reload_check")
